# Model Performance — Bar Charts

Loads computed metrics from `metrics.json` files written by `stress_predictions.ipynb`  
and generates the multi-site comparison bar chart figure.

**No manual metric entry required.** Run `stress_predictions.ipynb` first to generate the JSON files.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import json
from pathlib import Path

## Configuration

Must match the `EXP_NAME`, `SITES_TO_RUN`, and `TOLERANCE` used in `stress_predictions.ipynb`.

In [ ]:
EXP_NAME     = 'exp402'
OUTPUT_DIR   = Path('../output')
SITES_TO_RUN = ['sombrero', 'molasses', 'sk']
TOLERANCE    = 1  # Must match the tolerance used when running stress_predictions.ipynb

SITE_DISPLAY_NAMES = {
    'sombrero': 'Sombrero',
    'sk':       'Sand Key',
    'molasses': 'Molasses',
}

MODEL_NAMES   = ['XGB', 'Linear', 'Frequency']
MODEL_COLORS  = {'XGB': 'deeppink', 'Linear': 'silver', 'Frequency': 'green'}

FIGURE_OUTPUT = OUTPUT_DIR / EXP_NAME / f'all_sites_performance_{TOLERANCE}tol.png'

## Load metrics from JSON

In [ ]:
all_metrics = {}

for site in SITES_TO_RUN:
    json_path = OUTPUT_DIR / EXP_NAME / site / 'metrics.json'
    if not json_path.exists():
        raise FileNotFoundError(
            f'Metrics file not found: {json_path}\n'
            f'Run stress_predictions.ipynb for site "{site}" first.'
        )
    with open(json_path) as f:
        data = json.load(f)

    # Verify tolerance matches
    if data.get('tolerance') != TOLERANCE:
        print(f'Warning: {site} metrics were computed with tolerance={data["tolerance"]},'
              f' but TOLERANCE={TOLERANCE} is set here.')

    all_metrics[site] = data['models']
    site_display = data.get('site_display', site)
    print(f'{site_display}: '
          + '  '.join(f"{m} F1={all_metrics[site][m]["f1_score"]:.3f}" for m in MODEL_NAMES))

print('\nMetrics loaded successfully.')

## Multi-site bar chart

In [ ]:
site_display_list = [SITE_DISPLAY_NAMES.get(s, s) for s in SITES_TO_RUN]

# Extract arrays for each metric across sites and models
metric_keys = [
    ('onset_accuracy',    f'Onset Accuracy\n(±{TOLERANCE} week)'),
    ('no_onset_accuracy', 'No Onset Accuracy'),
    ('false_alarm_rate',  'False Alarm Rate'),
    ('missed_onset_rate', 'Missed Onset Rate'),
]

f1_data = {
    model: [all_metrics[site][model]['f1_score'] for site in SITES_TO_RUN]
    for model in MODEL_NAMES
}

fig = plt.figure(figsize=(14, 12))
gs  = fig.add_gridspec(3, 2, height_ratios=[1.2, 1, 1], hspace=0.55)

# ── F1 score panel (top, full width) ─────────────────────────────────────────
ax_f1 = fig.add_subplot(gs[0, :])
bar_width = 0.2
x = np.arange(len(SITES_TO_RUN))

for i, model in enumerate(MODEL_NAMES):
    ax_f1.bar(
        x + i * bar_width,
        f1_data[model],
        width=bar_width,
        label=model,
        color=MODEL_COLORS[model],
        edgecolor='black' if model == 'XGB' else None,
        zorder=3 if model == 'XGB' else 2,
    )

ax_f1.set_ylabel('F1 Score', fontsize=16)
ax_f1.set_title('Overall Performance (F1 Score)', fontsize=18, weight='bold')
ax_f1.set_xticks(x + bar_width)
ax_f1.set_xticklabels(site_display_list, fontsize=18)
ax_f1.set_ylim(0, 1)
ax_f1.grid(axis='y', linestyle='--', alpha=0.5)
ax_f1.tick_params(labelsize=16)
ax_f1.legend(
    title='Model', fontsize=20, title_fontsize=20,
    loc='center left', bbox_to_anchor=(1.07, 0.5), frameon=False,
)

# ── Metric subplots (2×2 grid) ────────────────────────────────────────────────
axes = [fig.add_subplot(gs[i // 2 + 1, i % 2]) for i in range(4)]

for idx, (metric_key, metric_label) in enumerate(metric_keys):
    ax = axes[idx]
    for i, model in enumerate(MODEL_NAMES):
        values = [all_metrics[site][model][metric_key] for site in SITES_TO_RUN]
        ax.bar(
            x + i * bar_width,
            values,
            width=bar_width,
            color=MODEL_COLORS[model],
            edgecolor='black' if model == 'XGB' else None,
            zorder=3 if model == 'XGB' else 2,
        )
    ax.set_title(metric_label, fontsize=16)
    ax.set_ylabel('Percentage (%)', fontsize=14)
    ax.set_xticks(x + bar_width)
    ax.set_xticklabels(site_display_list, fontsize=16)
    ax.set_ylim(0, 100)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.tick_params(labelsize=13)

fig.suptitle('Model Performance Across Reef Sites', fontsize=26, y=0.98)
plt.tight_layout(rect=[0, 0, 0.93, 0.97])

FIGURE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE_OUTPUT, dpi=300, bbox_inches='tight')
plt.show()
print(f'Figure saved to {FIGURE_OUTPUT}')

## Print metrics summary table

In [ ]:
print(f'Tolerance = ±{TOLERANCE} week\n')
header = f'{"Site":<12} {"Model":<12} {"OnsetAcc%":>10} {"NoOnsetAcc%":>12} {"FA%":>8} {"Miss%":>8} {"F1":>8}'
print(header)
print('-' * len(header))

for site in SITES_TO_RUN:
    display = SITE_DISPLAY_NAMES.get(site, site)
    for model in MODEL_NAMES:
        m = all_metrics[site][model]
        print(
            f'{display:<12} {model:<12}'
            f' {m["onset_accuracy"]:>10.1f}'
            f' {m["no_onset_accuracy"]:>12.1f}'
            f' {m["false_alarm_rate"]:>8.1f}'
            f' {m["missed_onset_rate"]:>8.1f}'
            f' {m["f1_score"]:>8.3f}'
        )
    print()